# 第四周练习 —— 传感器遥测模拟器（Sensor Telemetry Simulator）

## 练习目标

用 LLM **模拟 IoT 传感器遥测**（像经 MQTT 发往云端的 JSON），并附一段自然语言摘要；结果展示在 Gradio UI。

## 和本课第 4 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多后端客户端 | OpenAI 云端 + 本地 Ollama（OpenAI 兼容 `/v1`） |
| 模型 → 客户端映射 | `clients` 字典按模型名选对端点 |
| 结构化输出约定 | Prompt 要求 `TELEMETRY:` / `SUMMARY:` 分段，再 `split` 解析 |
| Gradio Blocks | 下拉选模型，一键生成 JSON + 摘要 |

## 怎么跑

1. `.env`：`OPENAI_API_KEY`、`HF_TOKEN`（`login` 用）；本地模型需 Ollama 在 `localhost:11434`
2. 确保已拉取 `deepseek-r1:1.5b`、`llama3.2`（若要用本地项）
3. 依次运行单元格，在 UI 点 **Generate telemetry**


In [ ]:
# 导入 os：读环境变量
import os
# io / sys：通用标准库（本格主要为后续扩展预留，保持原导入）
import io
import sys
# load_dotenv：从 .env 加载密钥到环境变量
from dotenv import load_dotenv
# OpenAI SDK：同时用于云端 OpenAI 与本地 Ollama 兼容接口
from openai import OpenAI
# Gradio：搭建遥测模拟 UI
import gradio as gr
# subprocess：进程相关（本笔记本主流程未直接用到，保留原导入）
import subprocess
# IPython 显示工具（本格导入保留）
from IPython.display import Markdown, display
# Hugging Face Hub 登录：用 HF_TOKEN 鉴权
from huggingface_hub import login


In [ ]:
# 加载 .env；override=True 覆盖已有同名环境变量
load_dotenv(override=True)

# 读取 OpenAI API Key
openai_api_key = os.getenv('OPENAI_API_KEY')
# 读取 Hugging Face Token（后面 login 用）
hf_token = os.getenv('HF_TOKEN')

# 有 key 时只打印前 8 位做存在性检查，避免泄露全文
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
# 登录 Hugging Face Hub；add_to_git_credential=True 按原参数保留
login(hf_token, add_to_git_credential=True)


In [ ]:
# 本地 Ollama 的 OpenAI 兼容端点（/v1）
ollama_url = "http://localhost:11434/v1"
# api_key 对本地 Ollama 常可填任意非空串；这里用字面 "ollama"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# 云端 OpenAI 客户端（默认 api.openai.com）
openai = OpenAI(api_key=openai_api_key)


In [ ]:
# 可选模型列表：前两项走 OpenAI，后两项走本地 Ollama
models = [
    "gpt-4o-mini",
    "gpt-4o",
    "deepseek-r1:1.5b",
    "llama3.2",
]

# 模型名 → 对应客户端实例（路由表）
clients = {
    "gpt-4o-mini": openai,
    "gpt-4o": openai,
    "deepseek-r1:1.5b": ollama,
    "llama3.2": ollama,
}


def get_client(model_name: str):
    """按模型名返回该用的 API 客户端（Ollama 或 OpenAI）。"""
    # 从字典取客户端；取不到则为 None
    client = clients.get(model_name)
    # 若映射到 ollama 实例，显式返回 ollama
    if client is ollama:
        return ollama
    # 其余情况（含未命中）走 openai 客户端
    return openai


In [ ]:
# user prompt：要求模型先输出 TELEMETRY JSON，再输出 SUMMARY；格式标记供后面 split（勿改译）
TELEMETRY_USER_PROMPT = """
Simulate telemetry from an IoT sensor as it would be sent via MQTT to a cloud service.

1. Generate a realistic JSON object with plausible sensor measurements (e.g. temperature, humidity, pressure, battery, timestamp, device_id).
2. Then write a brief natural-language summary interpreting the data, noting any trends or anomalies.

Format your response exactly as follows:
TELEMETRY:
<valid JSON object here>
SUMMARY:
<your brief summary here>
"""


In [ ]:
def generate_telemetry(model_name: str) -> tuple[str, str]:
    """调用选定模型，生成 MQTT 风格传感器遥测 JSON + 自然语言摘要。"""
    # 按模型名拿到对应客户端（云端或本地）
    client = get_client(model_name)
    # Chat Completions：单条 user 消息，即 TELEMETRY_USER_PROMPT
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": TELEMETRY_USER_PROMPT}],
        # 略提高随机性，让每次模拟读数有变化
        temperature=0.7,
    )
    # 取出回复文本；若 content 为 None 则用空串
    text = response.choices[0].message.content or ""
    # 预置两段结果
    telemetry_text, summary_text = "", ""
    # 按约定标记 SUMMARY: 拆成「遥测」与「摘要」
    if "SUMMARY:" in text:
        parts = text.split("SUMMARY:", 1)
        # 去掉 TELEMETRY: 前缀，留下 JSON（或模型原样输出）
        telemetry_text = parts[0].replace("TELEMETRY:", "").strip()
        summary_text = parts[1].strip()
    else:
        # 没有 SUMMARY: 时整段当作遥测，摘要留空
        telemetry_text = text
    # 返回给 Gradio 两个 TextArea
    return telemetry_text, summary_text


In [ ]:
# Soft 主题 + 标题；Blocks 作为整页 UI
with gr.Blocks(theme=gr.themes.Soft(), title="Sensor Telemetry Simulator") as ui:
    # 说明：模拟 IoT 遥测 + MQTT 语义（UI 英文保留）
    gr.Markdown("## Sensor Telemetry Simulator\nGenerate and interpret IoT sensor telemetry as would be sent via MQTT to a cloud service.")
    with gr.Row():
        # 模型下拉：默认优先 gpt-4o-mini（若在列表中）
        model = gr.Dropdown(choices=models, value="gpt-4o-mini" if "gpt-4o-mini" in models else models[0], label="Model")
        # 主按钮：触发生成
        generate_btn = gr.Button("Generate telemetry", variant="primary")
    with gr.Row():
        # 左：遥测 JSON 文本区
        telemetry_out = gr.TextArea(label="Telemetry data (JSON)", lines=12, placeholder="Generated sensor payload will appear here…")
        # 右：自然语言摘要
        summary_out = gr.TextArea(label="Summary", lines=12, placeholder="Natural-language interpretation will appear here…")
    # 点击：model → generate_telemetry → 两个输出框
    generate_btn.click(fn=generate_telemetry, inputs=[model], outputs=[telemetry_out, summary_out])

# 启动 Gradio，并尝试打开浏览器
ui.launch(inbrowser=True)
